<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install all required libraries for downloading market data, running PCA, and visualizing the results.

In [ ]:
!pip install yfinance pandas numpy scikit-learn matplotlib

## Imports and setup

We use yfinance to pull historical stock prices, pandas and numpy for data manipulation and linear algebra, scikit-learn's PCA class to extract hidden return drivers, and matplotlib to visualize the results.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

## Download stock data and compute returns

We define a mixed portfolio of tech stocks and gold miners, then download their daily closing prices and convert them to percentage returns. This is the raw material PCA will decompose into hidden drivers.

In [ ]:
symbols = [
    'IBM',
    'MSFT',
    'META',
    'INTC',
    'NEM',
    'AU',
    'AEM',
    'GFI'
]
data = yf.download(symbols, start="2020-01-01", end="2022-11-30")
portfolio_returns = data.Close.pct_change().dropna()

The portfolio deliberately mixes two sectors that seem unrelated on the surface. PCA will reveal whether these stocks actually move independently or share common forces underneath. Working with percentage returns rather than raw prices ensures we compare movements on the same scale.

## Fit PCA and visualize explained variance

We fit PCA with three components to find the top three statistical drivers of our portfolio's returns. Three is a practical starting point since most equity portfolios are dominated by just a few forces.

In [ ]:
pca = PCA(n_components=3)
pca.fit(portfolio_returns)

Extract how much variance each component explains and the component weight vectors themselves. These two outputs are the core of the entire analysis.

In [ ]:
pct = pca.explained_variance_ratio_
pca_components = pca.components_

The explained variance ratio tells us what fraction of total portfolio movement each hidden driver accounts for. If the first component explains 50% or more, it means a single force (often broad market direction) dominates our supposedly diversified portfolio.

Plot individual and cumulative variance contributions side by side so we can see at a glance how concentrated the portfolio's risk drivers are.

In [ ]:
cum_pct = np.cumsum(pct)
x = np.arange(1, len(pct) + 1, 1)

plt.subplot(1, 2, 1)
plt.bar(x, pct * 100, align="center")
plt.title('Contribution (%)')
plt.xlabel('Component')
plt.xticks(x)
plt.xlim([0, 4])
plt.ylim([0, 100])

plt.subplot(1, 2, 2)
plt.plot(x, cum_pct * 100, 'ro-')
plt.title('Cumulative contribution (%)')
plt.xlabel('Component')
plt.xticks(x)
plt.xlim([0, 4])
plt.ylim([0, 100])
plt.show()

The bar chart shows each component's standalone contribution while the line chart shows the running total. If the cumulative line reaches 70–80% by the second or third component, it confirms that only a few hidden forces explain most of what our eight stocks are doing. This is the quantitative answer to "am I actually diversified?"

## Compute factor returns and exposures

Project the original daily returns onto the principal components to get a daily time series for each hidden factor. This transforms our eight-stock return matrix into three factor return streams.

In [ ]:
X = np.asarray(portfolio_returns)

In [ ]:
factor_returns = X.dot(pca_components.T)

In [ ]:
factor_returns = pd.DataFrame(
    columns=["f1", "f2", "f3"],
    index=portfolio_returns.index,
    data=factor_returns
)

In [ ]:
factor_returns.head()

Each row in factor_returns represents how much each hidden driver moved on a given day. Professionals use these synthetic return series to track regime changes, build hedges, or attribute portfolio performance to specific drivers rather than individual stock picks.

Reshape the component weights into a stock-by-factor table so we can see how strongly each stock loads onto each hidden driver.

In [ ]:
factor_exposures = pd.DataFrame(
    index=["f1", "f2", "f3"],
    columns=portfolio_returns.columns,
    data=pca_components
).T

Plot each stock's loading on the first principal component. Stocks with similar bar heights are being pushed around by the same dominant force.

In [ ]:
factor_exposures.f1.sort_values().plot.bar()
plt.show()

The sorted bar chart makes it easy to spot which stocks respond most (and least) to the portfolio's primary driver. If all bars point the same direction with similar magnitude, our "diversified" portfolio is really just one big bet on a single force. Stocks with opposite signs would genuinely offset each other.

## Map stock clusters with a scatter plot

Plot each stock's exposure to the first two components on a 2D scatter plot. Stocks that land near each other share similar sensitivities to the top two hidden forces, meaning they tend to move together.

In [ ]:
labels = factor_exposures.index
data = factor_exposures.values
plt.scatter(data[:, 0], data[:, 1])
plt.xlabel('factor exposure of PC1')
plt.ylabel('factor exposure of PC2')

for label, x, y in zip(labels, data[:, 0], data[:, 1]):
    plt.annotate(
        label,
        xy=(x, y),
        xytext=(-20, 20),
        textcoords='offset points',
        arrowprops=dict(
            arrowstyle='->',
            connectionstyle='arc3,rad=0'
        )
    )
plt.show()

This scatter plot is the payoff of the entire analysis. Tight clusters reveal stocks that look different on the surface but behave the same underneath. If the tech names group together and the gold miners form a separate cluster, we have genuine diversification along at least one axis. If everything collapses into one blob, we know our portfolio is less diversified than we thought and we can take concrete steps to fix it.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.